<center>
    <h1>
    Preprocess
    </h1>
</center>

In [1]:
%%html

<!-- styles d'affichage -->
<style>
    .section_div {
        width:70%;
        height:1.5px;
        border:none;
        color:black;
        background-color:black;
        margin: auto;
        margin-top: 0px;
        margin-bottom: 0px;
    }

    .answer {
        color:blue;
    }

    .question {
        color:red;
    }

    .note {
        color:green;
        font-weight: bold;
    }
</style>

In [2]:
#
# assure le reload de src si modifications sont faite
#
%load_ext autoreload
%autoreload 2

In [3]:
#
# import utilitaires
#
%matplotlib inline

import audioread
import librosa as rosa
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from pathlib import Path
from pprint import pprint
from tqdm.notebook import tqdm

In [4]:
#
# import package develope pour le projet
#
import ffury
from ffury.configs import (
    DatasetType,
    DEFAULT_CONFIG_FILE,
    load_config
)

def get_config():
    # creation config - on sait que DEFAULT_CONFIG_FILE est dans le repertoire parent
    config = Path(ffury.__file__).parents[2].joinpath(DEFAULT_CONFIG_FILE)
    return load_config(config)

config = get_config()

# load dataset explore
explored_df = pd.read_csv(config.get_csv_filename(DatasetType.EXPLORED))

In [8]:
# validation des donnees explorees
print("Dataset shape:", explored_df.shape)

print()

print("Types")
display(explored_df.dtypes.to_frame().T)

print

print("Affichage succint")
display(explored_df.head())

Dataset shape: (2239, 6)

Types


,common_name,primary_label,latitude,longitude,filename,Duration
0,object,object,float64,float64,object,float64


Affichage succint


,common_name,primary_label,latitude,longitude,filename,Duration
0,African Black-headed Oriole,abhori1,-14.2421,33.4638,abhori1/XC348440.ogg,10.4
1,African Black-headed Oriole,abhori1,-32.5702,27.3153,abhori1/XC247351.ogg,14.8
2,African Bare-eyed Thrush,abethr1,4.8403,38.6988,abethr1/XC531557.ogg,19.9
3,African Black-headed Oriole,abhori1,-31.0664,30.1797,abhori1/XC442426.ogg,13.3
4,African Black-headed Oriole,abhori1,-2.9965,37.6244,abhori1/XC393448.ogg,18.0


<div class="answer">

* Shape est telle qu'attendue (voir 00-jfgagnon-exploration.ipynb/00-jfgagnon-exploration.pdf)
* Les attributs sont tel qu'attendus.

<div class="answer">

On veut préparer notre data pour le multi-instance learning pour l'audio. On doit donc séparer en segments puis en prendre une suite pour faire 1 classification. Ex. fichier audio de 10 secondes sera morceller en segments de 1 secondes qui se chevauche à tous les 0.5s. Une suite de 5 segements (longeur totale de 3 secondes à cause du chevauchement) sera donc 1 éléments à classifier. Une batch regroupera par exemple 64 séries de 5 segments consécutifs. On appellera cette série un groupe.

Noter que l'exploration n'a pas pris de décision sur le balancement des classes. Notre data final doit donc resampler les groupes disponibles afin d'avoir quelque chose de balancé par espèce d'oiseau.

Dernier élément ; on veut peut-être s'assurer qu'il y a présence d'audio intéressant. Peut-être faire un filtre par medianne pour voir s'il y a qqch.

Train/test/validation : on va séparer en groupe et c'est ça qu'on split.

Préalable:
1. Voir combien on aurait de groupes par espèce
2. Garder 10 espèces ; celles avec le plus de groupes
3. Etablir combien de groupes on veut par especes

Donc si on résume:
1. Faire log-melspectrogram fichier audio au complet
2. Etablir un nombre de frames par fichier audio (1 frame dure la longeur de l'overlap)
3. Prendre un fichier random et selectionner un frame random
   3.1 Retirer 5 frames consécutifs à partir de celui déterminé, ca nous donne un sample avec son label
   3.2 Filtre mediane et valider qu'on a une bonne couverture

Ou on y va plus brutte ;  nombre de groupes par fichier et c'est tout